In [1]:
import pandas as pd

df = pd.read_csv("data/raw/DataCoSupplyChainDataset.csv", encoding="latin-1")

print(df.shape)

df.head()

(180519, 53)


,Type,Days for shipping (real),Days for shipment (scheduled),Benefit per order,Sales per customer,Delivery Status,Late_delivery_risk,Category Id,Category Name,Customer City,...,Order Zipcode,Product Card Id,Product Category Id,Product Description,Product Image,Product Name,Product Price,Product Status,shipping date (DateOrders),Shipping Mode
0,DEBIT,3,4,91.250000,314.640015,Advance shipping,0,73,Sporting Goods,Caguas,...,NaN,1360,73,NaN,http://images.acmesports.sports/Smart+watch,Smart watch,327.75,0,2/3/2018 22:56,Standard Class
1,TRANSFER,5,4,-249.089996,311.359985,Late delivery,1,73,Sporting Goods,Caguas,...,NaN,1360,73,NaN,http://images.acmesports.sports/Smart+watch,Smart watch,327.75,0,1/18/2018 12:27,Standard Class
2,CASH,4,4,-247.779999,309.720001,Shipping on time,0,73,Sporting Goods,San Jose,...,NaN,1360,73,NaN,http://images.acmesports.sports/Smart+watch,Smart watch,327.75,0,1/17/2018 12:06,Standard Class
3,DEBIT,3,4,22.860001,304.809998,Advance shipping,0,73,Sporting Goods,Los Angeles,...,NaN,1360,73,NaN,http://images.acmesports.sports/Smart+watch,Smart watch,327.75,0,1/16/2018 11:45,Standard Class
4,PAYMENT,2,4,134.210007,298.250000,Advance shipping,0,73,Sporting Goods,Caguas,...,NaN,1360,73,NaN,http://images.acmesports.sports/Smart+watch,Smart watch,327.75,0,1/15/2018 11:24,Standard Class


In [2]:
df.columns.tolist()

['Type',
 'Days for shipping (real)',
 'Days for shipment (scheduled)',
 'Benefit per order',
 'Sales per customer',
 'Delivery Status',
 'Late_delivery_risk',
 'Category Id',
 'Category Name',
 'Customer City',
 'Customer Country',
 'Customer Email',
 'Customer Fname',
 'Customer Id',
 'Customer Lname',
 'Customer Password',
 'Customer Segment',
 'Customer State',
 'Customer Street',
 'Customer Zipcode',
 'Department Id',
 'Department Name',
 'Latitude',
 'Longitude',
 'Market',
 'Order City',
 'Order Country',
 'Order Customer Id',
 'order date (DateOrders)',
 'Order Id',
 'Order Item Cardprod Id',
 'Order Item Discount',
 'Order Item Discount Rate',
 'Order Item Id',
 'Order Item Product Price',
 'Order Item Profit Ratio',
 'Order Item Quantity',
 'Sales',
 'Order Item Total',
 'Order Profit Per Order',
 'Order Region',
 'Order State',
 'Order Status',
 'Order Zipcode',
 'Product Card Id',
 'Product Category Id',
 'Product Description',
 'Product Image',
 'Product Name',
 'Product P

In [6]:
df.columns = [c.strip() for c in df.columns]

In [7]:
carrier_map = {
    "Standard Class": "Carrier A (Ground)",
    "Second Class": "Carrier B (Expedited)",
    "First Class": "Carrier B (Expedited)",
    "Same Day": "Carrier C (Premium)",
}
df["carrier"] = df["Shipping Mode"].map(carrier_map).fillna("Carrier A (Ground)")

df["carrier"].value_counts()

carrier
Carrier A (Ground)       107752
Carrier B (Expedited)     63030
Carrier C (Premium)        9737
Name: count, dtype: int64

In [8]:
df["warehouse"] = df["Order Region"].astype(str).str.strip()

df["warehouse"].value_counts()

warehouse
Central America    28341
Western Europe     27109
South America      14935
Oceania            10148
Northern Europe     9792
Southeast Asia      9539
Southern Europe     9431
Caribbean           8318
West of USA         7993
South Asia          7731
Eastern Asia        7280
East of USA         6915
West Asia           6009
US Center           5887
South of  USA       4045
Eastern Europe      3920
West Africa         3696
North Africa        3232
East Africa         1852
Central Africa      1677
Southern Africa     1157
Canada               959
Central Asia         553
Name: count, dtype: int64

In [9]:
df["warehouse"].nunique()

23

In [10]:
df["warehouse"].value_counts()

warehouse
Central America    28341
Western Europe     27109
South America      14935
Oceania            10148
Northern Europe     9792
Southeast Asia      9539
Southern Europe     9431
Caribbean           8318
West of USA         7993
South Asia          7731
Eastern Asia        7280
East of USA         6915
West Asia           6009
US Center           5887
South of  USA       4045
Eastern Europe      3920
West Africa         3696
North Africa        3232
East Africa         1852
Central Africa      1677
Southern Africa     1157
Canada               959
Central Asia         553
Name: count, dtype: int64

SyntaxError: unmatched ')' (300496786.py, line 1)

In [12]:
warehouse_map = {
    # North America — 5 regions → 1
    "West of USA": "North America",
    "East of USA": "North America",
    "South of  USA": "North America",   # note: dataset has a double space here
    "US Center": "North America",
    "Canada": "North America",

    # Central America
    "Central America": "Central America",

    # South America
    "South America": "South America",

    # Caribbean
    "Caribbean": "Caribbean",

    # Europe — kept as 4 separate buckets (each has solid volume)
    "Western Europe": "Western Europe",
    "Northern Europe": "Northern Europe",
    "Southern Europe": "Southern Europe",
    "Eastern Europe": "Eastern Europe",

    # Asia — 5 regions → 2
    "Southeast Asia": "East & Southeast Asia",
    "Eastern Asia": "East & Southeast Asia",
    "South Asia": "South & Central Asia",
    "West Asia": "South & Central Asia",
    "Central Asia": "South & Central Asia",

    # Oceania
    "Oceania": "Oceania",

    # Africa — 5 regions → 1
    "West Africa": "Africa",
    "North Africa": "Africa",
    "East Africa": "Africa",
    "Central Africa": "Africa",
    "Southern Africa": "Africa",
}

df["warehouse"] = df["Order Region"].astype(str).str.strip().map(warehouse_map)

df["warehouse"].nunique(), df["warehouse"].value_counts()

(12,
 warehouse
 Central America          28341
 Western Europe           27109
 North America            25799
 East & Southeast Asia    16819
 South America            14935
 South & Central Asia     14293
 Africa                   11614
 Oceania                  10148
 Northern Europe           9792
 Southern Europe           9431
 Caribbean                 8318
 Eastern Europe            3920
 Name: count, dtype: int64)

In [13]:
df["days_scheduled"] = df["Days for shipment (scheduled)"]
df["days_actual"] = df["Days for shipping (real)"]
df["delay_days"] = df["days_actual"] - df["days_scheduled"]
df["sla_breach"] = (df["Late_delivery_risk"] == 1).astype(int)

df[["days_scheduled", "days_actual", "delay_days", "sla_breach"]].describe()

,days_scheduled,days_actual,delay_days,sla_breach
count,180519.000000,180519.000000,180519.000000,180519.000000
mean,2.931847,3.497654,0.565807,0.548291
std,1.374449,1.623722,1.490966,0.497664
min,0.000000,0.000000,-2.000000,0.000000
25%,2.000000,2.000000,0.000000,0.000000
50%,4.000000,3.000000,1.000000,1.000000
75%,4.000000,5.000000,1.000000,1.000000
max,4.000000,6.000000,4.000000,1.000000
